# Phase 9: Generalization Test (100 unseen VMs)

So far every model was tested on the *last week of the same 50 VMs* it trained on. A cloud provider, though, would use one model across many machines, including new ones. Here we take the trained models **exactly as they are** (no retraining, no adjustments) and ask them to predict the CPU of **100 VMs they have never seen**: the generalization group that Phase 1 set aside and nothing has touched since.

If accuracy stays close to the original test set, the models have learned general patterns of VM behaviour. If it drops a lot, they partly memorised the particular 50 VMs.

**What we evaluate:**
- **Phase 3 models (default settings):** the models the original plan asks for.
- **Phase 4 models (tuned settings):** also checked, because they're the versions TOPSIS ranks and recommends.
- **Persistence reference** (next CPU = current CPU), as before.

**Same rows for every model:** the LSTM needs 12 earlier readings before it can make its first prediction for a VM, so we score every model on the rows where all of them can predict. That only drops the first 11 rows of each VM.

**Run time:** about 2–4 minutes (loading the big Random Forest model takes the longest).

## 0. Setup

In [1]:
# Mount Google Drive (Colab) or use the local Drive folder (PC), then load our shared code.
import os, sys, importlib

try:
    from google.colab import drive
    drive.mount('/content/drive')
    PROJECT_ROOT = '/content/drive/MyDrive/Green_ML_Project'
except ImportError:
    PROJECT_ROOT = r'H:\My Drive\Green_ML_Project'   # running on the PC instead of Colab

os.environ['GREEN_ML_ROOT'] = PROJECT_ROOT
sys.path.insert(0, os.path.join(PROJECT_ROOT, 'scripts'))

import config as C
import utils as U
importlib.reload(C); importlib.reload(U)   # picks up any edits to the scripts without restarting

print('Project folder:', C.PROJECT_ROOT)
print('Folder found:  ', os.path.isdir(C.PROJECT_ROOT))

Mounted at /content/drive
Project folder: /content/drive/MyDrive/Green_ML_Project
Folder found:   True


In [2]:
import gc, json
import numpy as np
import pandas as pd
import joblib, torch
import model_definitions as M
importlib.reload(M)

pd.set_option('display.width', 180)
log = U.get_logger('phase09_generalization_test')

RUN STARTED 2026-10-04 07:29:33 | phase09_generalization_test
python 3.13.15 | pandas 2.2.3 | numpy 2.1.3 | machine: Linux-6.6.122+-x86_64-with-glibc2.39
project root: /content/drive/MyDrive/Green_ML_Project


## 1. Load the unseen VMs and line up the rows

In [3]:
gen = pd.read_parquet(C.DATA_PROCESSED / 'features_generalization.parquet')
test = pd.read_parquet(C.DATA_PROCESSED / 'test.parquet')
phase3 = json.load(open(C.MODELS_DIR / 'phase3_results.json'))
phase4 = json.load(open(C.MODELS_DIR / 'phase4_results.json'))

main_vms = set(pd.read_parquet(C.DATA_PROCESSED / 'train.parquet', columns=['vm_id'])['vm_id'])
assert not main_vms & set(gen['vm_id']), 'generalization VMs overlap with training VMs!'
log.info(f'Generalization set: {gen["vm_id"].nunique()} VMs, {len(gen):,} rows; overlap with training VMs: none')

# Rows where every model (including the LSTM with the longest history) can predict
lstm_seq = {'default': M.LSTM_DEFAULTS['seq_len'], 'tuned': phase4['LSTM']['best_params']['seq_len']}
max_seq = max(lstm_seq.values())
_, _, common_idx = M.make_sequences(gen, max_seq)
common = gen.loc[common_idx].reset_index(drop=True)
y_gen = common[C.TARGET].to_numpy()
log.info(f'Rows scored for every model: {len(common):,} ({len(gen) - len(common):,} dropped: first {max_seq - 1} per VM)')

Generalization set: 100 VMs, 816,878 rows; overlap with training VMs: none
Rows scored for every model: 815,778 (1,100 dropped: first 11 per VM)


## 2. Predict with every model

Each model is loaded from Drive exactly as saved in Phase 3 or Phase 4, then predicts all the common rows.

In [4]:
def lstm_predict(version):
    p = phase4['LSTM']['best_params'] if version == 'tuned' else {}
    hidden = p.get('hidden_size', M.LSTM_DEFAULTS['hidden_size'])
    seq = p.get('seq_len', M.LSTM_DEFAULTS['seq_len'])
    model = M.build_lstm(len(M.LSTM_SEQ_FEATURES), hidden, M.LSTM_DEFAULTS['num_layers'], C.RANDOM_SEED)
    model.load_state_dict(torch.load(C.MODELS_DIR / ('tuned_lstm.pt' if version == 'tuned' else 'lstm.pt')))
    X, y, idx = M.make_sequences(gen, seq)
    keep = pd.Index(idx).get_indexer(common_idx)          # pick the common rows, in the same order
    assert (keep >= 0).all()
    return M.predict_lstm(model, X[keep])

scores, preds = [], {}
for version, prefix in (('default', ''), ('tuned', 'tuned_')):
    for name in M.TABULAR_MODELS + ['LSTM']:
        with U.Timer() as t:
            if name == 'LSTM':
                pred = lstm_predict(version)
            else:
                path = C.MODELS_DIR / f'{prefix}{M.slug(name)}.joblib'
                if version == 'tuned' and name == 'Linear Regression':
                    path = C.MODELS_DIR / 'linear_regression.joblib'   # no tuning for LR
                model = joblib.load(path)
                pred = model.predict(common[U.FEATURE_COLUMNS])
                del model; gc.collect()
        s = U.regression_metrics(y_gen, pred)
        scores.append({'Model': name, 'Version': version, **s})
        preds[(name, version)] = pred
        log.info(f'{version:<7} | {name:<17} | MAE {s["MAE"]:.4f} | RMSE {s["RMSE"]:.4f} | R2 {s["R2"]:.4f} | {t.seconds:.1f}s')

persist = U.regression_metrics(y_gen, common['cpu_usage_pct'].to_numpy())
log.info('Persistence reference: ' + ' | '.join(f'{k} {v:.4f}' for k, v in persist.items()))
scores = pd.DataFrame(scores)
scores.to_csv(C.TABLES_DIR / 'phase9_generalization_raw.csv', index=False)

default | Linear Regression | MAE 0.7069 | RMSE 3.3503 | R2 0.9612 | 2.2s
default | Decision Tree     | MAE 2.2367 | RMSE 9.2770 | R2 0.7028 | 1.7s
default | Random Forest     | MAE 1.3281 | RMSE 4.9424 | R2 0.9156 | 26.5s
default | XGBoost           | MAE 0.8830 | RMSE 3.3892 | R2 0.9603 | 2.5s
default | LightGBM          | MAE 0.9335 | RMSE 4.4142 | R2 0.9327 | 6.9s
default | LSTM              | MAE 0.5966 | RMSE 3.2490 | R2 0.9635 | 25.7s
tuned   | Linear Regression | MAE 0.7069 | RMSE 3.3503 | R2 0.9612 | 0.4s
tuned   | Decision Tree     | MAE 1.5813 | RMSE 5.8048 | R2 0.8836 | 0.7s
tuned   | Random Forest     | MAE 1.0949 | RMSE 4.6015 | R2 0.9269 | 4.2s
tuned   | XGBoost           | MAE 0.9193 | RMSE 3.3845 | R2 0.9604 | 4.0s
tuned   | LightGBM          | MAE 1.0827 | RMSE 4.4756 | R2 0.9308 | 5.3s
tuned   | LSTM              | MAE 0.6959 | RMSE 3.2350 | R2 0.9639 | 11.5s
Persistence reference: MAE 0.5055 | RMSE 3.3154 | MAPE 12.4098 | R2 0.9620


## 3. Original test set vs unseen VMs

**Change (%)** compares accuracy on the unseen VMs with accuracy on the original test set:
- For MAE and RMSE, a **positive** change means bigger errors on unseen VMs (worse).
- For R², a **negative** change means worse.

In [5]:
def comparison(version, source):
    rows = []
    for name in M.TABULAR_MODELS + ['LSTM']:
        orig = source.get(name, phase3[name])
        new = scores[(scores.Model == name) & (scores.Version == version)].iloc[0]
        rows.append({'Model': name,
                     'Test MAE': orig['MAE'], 'Unseen MAE': new.MAE, 'MAE change (%)': 100 * (new.MAE - orig['MAE']) / orig['MAE'],
                     'Test RMSE': orig['RMSE'], 'Unseen RMSE': new.RMSE, 'RMSE change (%)': 100 * (new.RMSE - orig['RMSE']) / orig['RMSE'],
                     'Test R2': orig['R2'], 'Unseen R2': new.R2, 'R2 change (%)': 100 * (new.R2 - orig['R2']) / orig['R2']})
    return pd.DataFrame(rows)

comp_default = comparison('default', phase3)
comp_tuned = comparison('tuned', phase4)
U.log_df(log, comp_default.round(4), 'GENERALIZATION: PHASE 3 MODELS (default settings)')
U.log_df(log, comp_tuned.round(4), 'GENERALIZATION: PHASE 4 MODELS (tuned settings)')
U.save_table(comp_default, 'phase9_generalization_default', 'Original test set vs 100 unseen VMs (default models)')
U.save_table(comp_tuned, 'phase9_generalization_tuned', 'Original test set vs 100 unseen VMs (tuned models)')
comp_default.round(4)

--- GENERALIZATION: PHASE 3 MODELS (default settings) ---
               Model  Test MAE  Unseen MAE  MAE change (%)  Test RMSE  Unseen RMSE  RMSE change (%)  Test R2  Unseen R2  R2 change (%)
0  Linear Regression    0.9032      0.7069        -21.7314     4.5899       3.3503         -27.0071   0.9062     0.9612         6.0677
1      Decision Tree    0.8468      2.2367        164.1319     5.2473       9.2770          76.7972   0.8775     0.7028       -19.9097
2      Random Forest    0.6903      1.3281         92.3834     3.6853       4.9424          34.1106   0.9396     0.9156        -2.5461
3            XGBoost    0.7007      0.8830         26.0227     3.7376       3.3892          -9.3206   0.9378     0.9603         2.3987
4           LightGBM    0.6743      0.9335         38.4372     3.7123       4.4142          18.9089   0.9387     0.9327        -0.6357
5               LSTM    0.8774      0.5966        -32.0048     4.5345       3.2490         -28.3483   0.9085     0.9635         6.05

,Model,Test MAE,Unseen MAE,MAE change (%),Test RMSE,Unseen RMSE,RMSE change (%),Test R2,Unseen R2,R2 change (%)
0,Linear Regression,0.9032,0.7069,-21.7314,4.5899,3.3503,-27.0071,0.9062,0.9612,6.0677
1,Decision Tree,0.8468,2.2367,164.1319,5.2473,9.2770,76.7972,0.8775,0.7028,-19.9097
2,Random Forest,0.6903,1.3281,92.3834,3.6853,4.9424,34.1106,0.9396,0.9156,-2.5461
3,XGBoost,0.7007,0.8830,26.0227,3.7376,3.3892,-9.3206,0.9378,0.9603,2.3987
4,LightGBM,0.6743,0.9335,38.4372,3.7123,4.4142,18.9089,0.9387,0.9327,-0.6357
5,LSTM,0.8774,0.5966,-32.0048,4.5345,3.2490,-28.3483,0.9085,0.9635,6.0590


In [6]:
comp_tuned.round(4)

,Model,Test MAE,Unseen MAE,MAE change (%),Test RMSE,Unseen RMSE,RMSE change (%),Test R2,Unseen R2,R2 change (%)
0,Linear Regression,0.9032,0.7069,-21.7314,4.5899,3.3503,-27.0071,0.9062,0.9612,6.0677
1,Decision Tree,1.0502,1.5813,50.5667,4.3561,5.8048,33.2547,0.9156,0.8836,-3.4870
2,Random Forest,0.6966,1.0949,57.1786,3.7493,4.6015,22.7294,0.9374,0.9269,-1.1275
3,XGBoost,0.6974,0.9193,31.8170,3.7151,3.3845,-8.8988,0.9386,0.9604,2.3292
4,LightGBM,0.6640,1.0827,63.0443,3.6507,4.4756,22.5941,0.9407,0.9308,-1.0490
5,LSTM,0.9238,0.6959,-24.6709,4.4939,3.2350,-28.0137,0.9101,0.9639,5.9036


## 4. Does the model order hold on unseen VMs?

If the models rank in the same order by MAE on the test set and on the unseen VMs, the conclusions from Phases 3–8 carry over to new machines.

We also look at the spread **across VMs**: the median MAE per VM and the worst VM. An average can hide a few VMs where a model fails badly.

In [7]:
for label, comp in (('default', comp_default), ('tuned', comp_tuned)):
    r_test = comp.set_index('Model')['Test MAE'].rank()
    r_gen = comp.set_index('Model')['Unseen MAE'].rank()
    rho = r_test.corr(r_gen)
    log.info(f'{label}: MAE ranking test vs unseen -> Spearman rho = {rho:.3f}; '
             f'best on test = {r_test.idxmin()}, best on unseen = {r_gen.idxmin()}')

per_vm = []
for (name, version), pred in preds.items():
    err = pd.Series(np.abs(pred - y_gen)).groupby(common['vm_id'].values).mean()
    per_vm.append({'Model': name, 'Version': version, 'Median VM MAE': err.median(),
                   'P90 VM MAE': err.quantile(0.9), 'Worst VM MAE': err.max(), 'Worst VM id': int(err.idxmax())})
per_vm = pd.DataFrame(per_vm)
per_vm.to_csv(C.TABLES_DIR / 'phase9_per_vm_spread.csv', index=False)
U.log_df(log, per_vm.round(4), 'PER-VM ERROR SPREAD ON UNSEEN VMs')

# Predictions on unseen VMs, saved for possible charts in Phase 10
out = common[['vm_id', 'timestamp', C.TARGET]].rename(columns={C.TARGET: 'actual'}).copy()
for (name, version), pred in preds.items():
    out[f'{M.slug(name)}_{version}'] = pred.astype('float32')
out.to_parquet(C.DATA_PROCESSED / 'generalization_predictions.parquet', index=False)
log.info('Saved: data/processed/generalization_predictions.parquet')
log.info('PHASE 9 COMPLETE')
per_vm.round(4)

default: MAE ranking test vs unseen -> Spearman rho = -0.543; best on test = LightGBM, best on unseen = LSTM
tuned: MAE ranking test vs unseen -> Spearman rho = -0.086; best on test = LightGBM, best on unseen = LSTM
--- PER-VM ERROR SPREAD ON UNSEEN VMs ---
                Model  Version  Median VM MAE  P90 VM MAE  Worst VM MAE  Worst VM id
0   Linear Regression  default         0.2583      2.1807        5.2705          256
1       Decision Tree  default         0.2733      4.4444       42.7499          763
2       Random Forest  default         0.2445      3.1467       31.5926          387
3             XGBoost  default         0.2167      2.8786        7.2708          763
4            LightGBM  default         0.1618      2.2599       30.3097          387
5                LSTM  default         0.1970      1.7806        5.0304          256
6   Linear Regression    tuned         0.2583      2.1807        5.2705          256
7       Decision Tree    tuned         0.5613      3.0117     

,Model,Version,Median VM MAE,P90 VM MAE,Worst VM MAE,Worst VM id
0,Linear Regression,default,0.2583,2.1807,5.2705,256
1,Decision Tree,default,0.2733,4.4444,42.7499,763
2,Random Forest,default,0.2445,3.1467,31.5926,387
3,XGBoost,default,0.2167,2.8786,7.2708,763
4,LightGBM,default,0.1618,2.2599,30.3097,387
5,LSTM,default,0.1970,1.7806,5.0304,256
6,Linear Regression,tuned,0.2583,2.1807,5.2705,256
7,Decision Tree,tuned,0.5613,3.0117,36.6333,387
8,Random Forest,tuned,0.1818,2.6684,29.7124,387
9,XGBoost,tuned,0.1918,2.9339,10.5262,591


## Done: what to check

- **Change (%)** columns: small changes (a few percent) mean the models generalize well. Note that the unseen VMs cover the *whole month*, while the original test set is only its *last week*, so some difference is expected just from that.
- **Spearman rho** close to 1 means the model order is the same on unseen VMs, so the TOPSIS recommendation still holds there.
- **Worst VM MAE**: if one VM is far worse than the median, it's worth a sentence in the Discussion (some workloads are simply unpredictable).

Next: `10_visualization_export.ipynb` makes every chart and table for the paper.